In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
import os

BASE = "/content/drive/MyDrive/hate_speech_ft"
os.makedirs(BASE, exist_ok=True)
os.makedirs(f"{BASE}/outputs", exist_ok=True)
os.makedirs(f"{BASE}/hf_cache", exist_ok=True)

os.environ["HF_HOME"] = f"{BASE}/hf_cache"
os.environ["HF_HUB_CACHE"] = f"{BASE}/hf_cache/hub"

In [ ]:
import os

REPO_URL = "https://github.com/Naaeeen/hate-speech-ft.git"
REPO_DIR = "/content/hate-speech-ft"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only

%cd $REPO_DIR

In [ ]:
!git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR

In [ ]:
import os

CACHE_DIR = f"{BASE}/pip_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
!pip install -q -r requirements-colab.txt --cache-dir $CACHE_DIR

In [ ]:
import torch
import transformers
import datasets
import peft
import wandb
import ipywidgets

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("wandb:", wandb.__version__)
print("ipywidgets:", ipywidgets.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import os
import wandb

WANDB_SECRET_NAME = "WANDB_API_KEY"

try:
    from google.colab import userdata
    wandb_api_key = userdata.get(WANDB_SECRET_NAME)
except Exception:
    wandb_api_key = None

if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login(key=wandb_api_key, relogin=True)
else:
    print(
        "No WANDB_API_KEY found in Colab Secrets. "
        "For online W&B logging, add it in Colab Secrets or run wandb.login(). "
        "You can still choose offline or disabled mode below."
    )


In [ ]:
!python src/utils/env_check.py
!python src/data/load_hatexplain.py
!python src/data/check_dataset.py

In [ ]:
!python src/run_experiment.py --validate_protocol
!python src/run_experiment.py --list --include_planned
!python src/run_experiment.py --experiment distilbert_full_smoke --dry_run

In [ ]:
from IPython.display import display
from src.colab.experiment_launcher import ExperimentLauncher

# Main entry point for smoke runs, HPO trial commands, confirmation seeds,
# final seeds, W&B logging, and local result aggregation.
launcher = ExperimentLauncher(
    default_entity="",
    default_project="hate-speech-ft",
)

display(launcher.view)


In [ ]:
from IPython.display import display

selected_config = launcher.get_config()
display(selected_config)
launcher.preview_command()


In [ ]:
launcher.run()


In [ ]:
# After HPO or final-seed runs finish, preview the aggregation command.
launcher.preview_aggregate_command()


In [ ]:
aggregate_report = launcher.aggregate_results()
aggregate_report["groups"][:5]
